In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, current_timestamp
from delta.tables import DeltaTable

print("CREATING DIM_AIRLINE FROM SILVER LAYER")

silver_airline_path = "s3://travel-analytics-bronze/delta/silver/airlines/"
gold_dim_airline_path = "s3://travel-analytics-bronze/delta/gold/Dim_Airline/"

# =============================================================
# STEP 1: LOAD SILVER DATA (Already Transformed)
# =============================================================
print("\nSTEP 1: Loading Silver Airlines Data...")

silver_df = spark.read.format("delta").load(silver_airline_path)

print(f"Loaded {silver_df.count():,} airline records from silver")

# =============================================================
# STEP 2: ADD DERIVED COLUMNS & SCD TYPE 2 METADATA
# =============================================================
print("\nSTEP 2: Creating Dimension Structure")
dim_airline_df = (
    silver_df
    
    # Add surrogate key
    .withColumn("Dim_Airline_SK", monotonically_increasing_id() + 1)

    # Select & rename columns to match dimension model
    .select(
        col("Dim_Airline_SK"),                       # PK
        col("Airline_Id").alias("Airline_ID_BK"),    # BK
        col("Airline_Name"),
        col("Country"),
        col("Airline_ICAO").alias("Airline_Icao"),
        col("Airline_IATA").alias("Airline_lata"),
        col("Fleet_Size"),
        col("Alias")
    )
)


CREATING DIM_AIRLINE FROM SILVER LAYER

STEP 1: Loading Silver Airlines Data...
Loaded 1,251 airline records from silver

STEP 2: Creating Dimension Structure


In [0]:
dim_airline_df.display()

Dim_Airline_SK,Airline_ID_BK,Airline_Name,Country,Airline_Icao,Airline_lata,Fleet_Size,Alias
1,5156,Tam Mercosur,Paraguay,LAP,PZ,16,UNKNOWN
2,18621,Ukraine Atlantic,Ukraine,UAT,UNKNOWN,6,UNKNOWN
3,1983,Darwin Airline,Switzerland,DWT,0D,8,UNKNOWN
4,3326,Lufttransport,Norway,LTR,L5,6,UNKNOWN
5,29,Askari Aviation,Pakistan,AAS,4K,6,UNKNOWN
6,569,Air India Express,India,AXB,IX,25,UNKNOWN
7,4573,Sunexpress,Turkey,SXS,XQ,58,UNKNOWN
8,13217,Syrian Pearl Airlines,Syria,PSB,UNKNOWN,4,UNKNOWN
9,18863,Russian Airline 18863,Russia,PKV,UNKNOWN,8,Pkv Airlines
10,3674,Nok Air,Thailand,NOK,DD,28,UNKNOWN


In [0]:
# =============================================================
# STEP 3: SAVE TO GOLD LAYER
# =============================================================
print("\nSTEP 3: Saving to Gold Layer...")
dim_airline_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_dim_airline_path)

print(f"Successfully created Dim_Airline with {dim_airline_df.count():,} records")


STEP 3: Saving to Gold Layer...
Successfully created Dim_Airline with 1,251 records
